# Create new Salesforce Job__c

POST a new `Job__c` record.

You can either:
- **Create from Supabase** `job_current` (set `USE_SUPABASE = True` and `SUPABASE_JOB_ID`), or
- **Create a brand-new test job without Supabase** (set `USE_SUPABASE = False` and edit `MANUAL_JOB_ROW`).

**Test markers:** Set `PROXI_SF_TEST_MODE=true` in `.env` to apply `[TEST]` / `[TEST RECORD]` prefixes. In test mode we also force **unique** values for `External_Job_ID__c` *and* `Job_Client_Job_Id__c` (some orgs enforce uniqueness on both), so you can re-run safely.

- `DRY_RUN = True` previews the payload without writing.

In [1]:
import os, sys, json
from pathlib import Path

project_root = Path.cwd().resolve()
for _ in range(15):
    if (project_root / "src" / "utils").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not locate repo root containing src/utils")

sys.path.insert(0, str(project_root / "src"))
from dotenv import load_dotenv
load_dotenv(project_root / ".env")
print("Ready")

Ready


In [2]:
# --- Change these ---
# Option A: Create using a real Supabase job_current row
USE_SUPABASE = False
SUPABASE_JOB_ID = "19596"
SUPABASE_SCHEMA = "public"

# Option B: Create a brand-new Job__c without Supabase
# - Set job_id to "AUTO" to generate a unique id each run (recommended).
# - Keep practice_value in the "2174 - Farmington, MO" format.
MANUAL_JOB_ROW = {
    "job_id": "AUTO",
    "city": "Farmington",
    "state": "MO",
    "practice_value": "2174 - Farmington, MO",
    "status": "Open",
    "insight": "",
    "dates_needed": "",
    "standard_schedule": "",
    "types_of_cases": "",
    "support_staff": "",
    "avg_patients_per_day": "",
    "roster_only": "false",
    "job_ranking": "B",
    "description_full_text": "",
    "point_of_contact": "",
    "address_line": "",
}

DRY_RUN = False

In [3]:
from datetime import datetime

from utils.supabase_db import load_job_current_row_for_salesforce


def _auto_job_id() -> str:
    # Keep it short so External_Job_ID__c remains within org limits after TEST- prefix.
    # Example: 240417173012 (YYMMDDhhmmss)
    return datetime.utcnow().strftime("%y%m%d%H%M%S")


if USE_SUPABASE:
    job_row = load_job_current_row_for_salesforce(SUPABASE_JOB_ID, schema=SUPABASE_SCHEMA)
else:
    job_row = dict(MANUAL_JOB_ROW)
    jid = str(job_row.get("job_id") or "").strip()
    if jid.upper() == "AUTO" or not jid:
        job_row["job_id"] = _auto_job_id()

print(
    f"Using job_id={job_row.get('job_id')}  city={job_row.get('city')}  state={job_row.get('state')}  "
    f"practice_value={job_row.get('practice_value')}"
)

Using job_id=260417090339  city=Farmington  state=MO  practice_value=2174 - Farmington, MO


In [4]:
from utils.salesforce import get_token_auto

token = get_token_auto(
    os.environ["SALESFORCE_CONSUMER_KEY"],
    os.environ["SALESFORCE_CONSUMER_SECRET"],
    os.environ.get("SALESFORCE_USERNAME") or None,
    os.environ.get("SALESFORCE_PASSWORD") or None,
    use_client_credentials=os.environ.get("SALESFORCE_USE_USERNAME_PASSWORD", "").lower() not in ("1", "true", "yes"),
    security_token=os.environ.get("SALESFORCE_SECURITY_TOKEN") or None,
    use_sandbox=os.environ.get("SALESFORCE_USE_SANDBOX", "").lower() in ("1", "true", "yes"),
    token_url=os.environ.get("SALESFORCE_TOKEN_URL") or "https://proxi.my.salesforce.com",
)
INSTANCE_URL = token["instance_url"]
ACCESS_TOKEN = token["access_token"]
print("Authenticated:", INSTANCE_URL)

Authenticated: https://proxi.my.salesforce.com


In [5]:
from utils.sf_job_payload import prepare_payload_for_write
from utils.sf_job_rest_minimal import describe_sobject

JOB_OBJECT = os.environ.get("SALESFORCE_JOB_OBJECT", "Job__c").strip()
describe = describe_sobject(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT)

fields = prepare_payload_for_write(
    job_row,
    describe,
    use_canonical_description=True,
    for_update=False,
    description_use_html=True,
)

# Optional test-only mutations (production pipeline uses the same flag in PATCH sync).
if os.environ.get("PROXI_SF_TEST_MODE", "").lower() in ("1", "true", "yes"):
    import uuid

    TEST_PREFIX = "[TEST] "
    if "Name" in fields:
        fields["Name"] = TEST_PREFIX + (fields["Name"] or "")

    # External_Job_ID__c must be unique (org enforces uniqueness). Generate a short stable-ish id
    # so you can re-run creates without colliding.
    suffix = uuid.uuid4().hex[:6].upper()
    if "External_Job_ID__c" in fields:
        base = str(fields.get("External_Job_ID__c") or "").strip()
        base_digits = "".join(ch for ch in base if ch.isdigit())[-10:]
        fields["External_Job_ID__c"] = f"T{suffix}{base_digits}"  # <= 17 chars

    # Some orgs also enforce uniqueness on Job_Client_Job_Id__c.
    # Keep the readable practice_value, but append a short test suffix.
    if "Job_Client_Job_Id__c" in fields and fields.get("Job_Client_Job_Id__c"):
        fields["Job_Client_Job_Id__c"] = f"{fields['Job_Client_Job_Id__c']} (T{suffix})"

    if "Job_Client_Job_Description__c" in fields:
        fields["Job_Client_Job_Description__c"] = "[TEST RECORD] " + (fields["Job_Client_Job_Description__c"] or "")
else:
    print("PROXI_SF_TEST_MODE not enabled — payload is go-live shaped (no TEST prefixes).")

print(f"{len(fields)} fields:", sorted(fields.keys()))

Skipped (not createable on object): Occupation_DJC__c
17 fields: ['External_Job_ID__c', 'External_Job_Link__c', 'Job_Account__c', 'Job_City__c', 'Job_Client_Job_Description__c', 'Job_Client_Job_Id__c', 'Job_Contract__c', 'Job_Job_Source__c', 'Job_Patient_Ages__c', 'Job_Position_Type__c', 'Job_Primary_Contact__c', 'Job_Ranking__c', 'Job_Specialty__c', 'Job_State__c', 'Job_Status__c', 'Salary_Pay_Range__c', 'roster_only__c']


In [6]:
from utils.sf_job_rest_minimal import create_job_record

show = dict(fields)
dk = "Job_Client_Job_Description__c"
if dk in show and len(str(show[dk])) > 500:
    show[dk] = str(show[dk])[:500] + f"... ({len(str(fields[dk]))} chars)"
print(json.dumps(show, indent=2, default=str))

if DRY_RUN:
    print("\nDRY_RUN — set DRY_RUN = False in cell 2 and re-run from there.")
else:
    result = create_job_record(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT, fields)
    new_id = result.get("id", "(unknown)")
    print(f"\nCreated {JOB_OBJECT} → {new_id}")

{
  "External_Job_ID__c": "T8032B90417090339",
  "Job_Account__c": "0015f00000HH63kAAD",
  "Job_Client_Job_Id__c": "2174 - Farmington, MO (T8032B9)",
  "Job_Client_Job_Description__c": "[TEST RECORD] <p><strong>General Dentist Locum Tenens Opportunity in Farmington, MO</strong></p><p><br/></p><p>We are seeking a General Dentist for a locum tenens opportunity in Farmington, Missouri. This position offers the opportunity to practice comprehensive general dentistry with a supportive clinical team and steady patient flow.</p><p><br/></p><p>Travel and lodging may be available for qualified candidates.</p><p><strong>Pay Range:</strong> Starting at $125/hour</p><p><br/></p><p><strong>... (883 chars)",
  "External_Job_Link__c": "https://portal.kimedics.com/app/workspace/job-posts/260417090339",
  "Job_Status__c": "Open",
  "Job_State__c": "Missouri",
  "Job_City__c": "Farmington",
  "Salary_Pay_Range__c": "Starting at $125/hour",
  "roster_only__c": "false",
  "Job_Position_Type__c": "Locums",